# Week 2: Modeling I, Linear Models, Product Mix, Multiperiod Planning and Inventory

**Course:** 2105623 Optimization of Chemical Processes  
**Institution:** Department of Chemical Engineering, Chulalongkorn University  
**Instructor:** Assoc. Prof. Dr. Soorathep Kheawhom

**Week:** 2 of 15 (modeling block, Weeks 1 to 6)  
**CLO mapping:** CLO 1 (formulation), CLO 4 (implementation and solution in Pyomo)

## Learning objectives

By the end of this notebook you should be able to:

- Carry a linear model through the five stages of the Week 1 workflow, from a verbal statement to an interpreted solution, and produce the artifact each stage requires.
- Name and apply the four reusable patterns of linear model construction (index sets, balance equations, capacity constraints, linking constraints) instead of inventing algebra problem by problem.
- Index decision variables over a discrete time horizon, write an inventory balance that links consecutive periods, and price backlog separately from held stock without introducing a binary variable.
- Extend a single-product plan to several products competing for one shared resource, and quantify what the coupling costs.
- Read the marginal value of a shared resource from the LP dual, verify it by finite difference, and use it to justify a debottlenecking decision period by period.

**Estimated duration:** 100 minutes  
**Prerequisites:** Week 1 (the five-stage workflow, the tool stack, model classification, first Pyomo formulation), linear algebra, basic cost accounting.

**Reference:** Rao, Ch. 3; Williams, *Model Building in Mathematical Programming*, Ch. 3 and Ch. 4 (product mix and multiperiod models).


In [ ]:
# --- Environment check -------------------------------------------------------
import sys, subprocess, importlib, shutil

def ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg])

for p, n in [("pyomo", "pyomo"), ("numpy", "numpy"), ("scipy", "scipy"),
             ("matplotlib", "matplotlib"), ("pandas", "pandas")]:
    ensure(p, n)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pyomo.environ as pyo

def pick_solver(kind="lp"):
    """Return the first available solver of the requested kind."""
    order = {"lp":   ["appsi_highs", "glpk", "cbc", "gurobi", "cplex"],
             "milp": ["appsi_highs", "cbc", "glpk", "gurobi", "cplex"],
             "nlp":  ["ipopt", "conopt", "knitro"],
             "minlp":["bonmin", "couenne", "mindtpy"]}[kind]
    for name in order:
        try:
            s = pyo.SolverFactory(name)
            if s is not None and s.available(exception_flag=False):
                print(f"Using solver: {name}")
                return s
        except Exception:
            continue
    raise RuntimeError(f"No {kind} solver found. Install one, e.g. 'pip install highspy' "
                       f"or 'conda install -c conda-forge ipopt glpk coincbc'.")

In [ ]:
# --- Figure style: Teal-Amber Lab Palette v1.0 -------------------------------
PALETTE = ["#0F6E6B", "#E29A2D", "#BE654C", "#5A91BE", "#83A462", "#995A90", "#333F4A", "#DFC98F"]
INK, GRAPHITE, MIST, PAPER = "#1C242B", "#333F4A", "#B9C1C6", "#F3F0EB"

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GRAPHITE, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.linewidth": 1.0, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": MIST, "grid.linewidth": 0.7, "grid.alpha": 0.9,
    "xtick.color": GRAPHITE, "ytick.color": GRAPHITE,
    "text.color": INK, "lines.linewidth": 1.8, "lines.markersize": 5,
    "font.size": 9, "legend.frameon": False,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
})

def tidy(ax):
    """Apply the house style to a single Axes object."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRAPHITE)
    return ax

print("Palette loaded:", ", ".join(PALETTE[:3]), "...")

## 1. Warm-up: one linear model in five stages

Week 1 introduced the workflow as a pipeline of five stages: verbal problem, structured specification, algebraic model, code, solution and interpretation. This section runs the whole pipeline once on a small product-mix problem, so that the format is fixed before the models get larger. Every model in Weeks 2 to 5 is presented this way.

### Stage 1: the verbal problem

A finishing shop attached to the electrolyte plant packages two grades of electrolyte for zinc-air cell assembly: a **Standard** grade and a **HighPurity** grade sold to a cell developer. Both grades pass through the same three operations, in order: mixing, ion-exchange purification, and filling. The three operations have different weekly hour allocations because they are shared with other campaigns. The HighPurity grade is worth more per tonne but spends four times as long in purification. Everything the shop finishes is sold. The shop supervisor must decide how many tonnes of each grade to finish in the coming week.

### Stage 2: the structured specification

Sets, data, decisions, one goal, and a numbered list of restrictions, each with units. This is written before any algebra.

| Object | Symbol | Meaning | Units |
|---|---|---|---|
| index set | `p in P` | grade, `P = {Standard, HighPurity}` | - |
| index set | `s in S` | operation, `S = {mixing, purification, filling}` | - |
| parameter | `c_p` | contribution margin of grade `p` | USD/t |
| parameter | `a_sp` | time on operation `s` per tonne of grade `p` | h/t |
| parameter | `b_s` | hours of operation `s` available this week | h/week |
| decision | `x_p >= 0` | tonnes of grade `p` finished this week | t/week |

Restrictions, in words:

- R1. On each operation, the hours consumed by the plan cannot exceed the hours allocated.
- R2. Production cannot be negative.
- R3. There is no demand limit, because everything finished is sold. (This is an assumption, and it is recorded here so that it can be challenged. Exercise 1 removes it.)

### Stage 3: the algebraic model

    maximize    z = sum_p c_p x_p                             [USD/week]

    subject to  sum_p a_sp x_p <= b_s     for all s in S      [h/week]
                x_p >= 0                  for all p in P

Units check, which is the whole reason stage 3 exists as a separate stage: `(h/t) * (t/week) = h/week` on the left of R1 and `h/week` on the right, and `(USD/t) * (t/week) = USD/week` in the objective. Both balance, so the algebra is at least dimensionally admissible.

The objective and all constraints are linear in `x`, and `x` is continuous, so this is a linear program in the sense of Week 1, Section 8. Stages 4 and 5 follow in the next cell.


In [ ]:
# --- Stage 4: data and code, then stage 5: solve, check, interpret -----------
GRADES = ["Standard", "HighPurity"]
OPS    = ["mixing", "purification", "filling"]

margin0 = pd.Series({"Standard": 520.0, "HighPurity": 790.0}, name="USD_per_t")
use0 = pd.DataFrame([[2.0, 3.0],      # mixing,       h/t
                     [1.0, 4.0],      # purification, h/t
                     [1.0, 1.0]],     # filling,      h/t
                    index=OPS, columns=GRADES)
avail0 = pd.Series({"mixing": 120.0, "purification": 104.0, "filling": 44.0},
                   name="h_per_week")

print("contribution margin, USD/t\n", margin0.to_string(), "\n")
print("operation time, h/t\n", use0.to_string(), "\n")
print("operation hours available, h/week\n", avail0.to_string())


def build_warmup(avail_h=None):
    """Stage 4: the algebraic model of Section 1, component by component."""
    bb = avail0 if avail_h is None else pd.Series(avail_h)
    m = pyo.ConcreteModel(name="warmup_product_mix")
    m.P = pyo.Set(initialize=GRADES, doc="grades")                      # index set
    m.S = pyo.Set(initialize=OPS, doc="operations")                     # index set
    m.c = pyo.Param(m.P, initialize=margin0.to_dict())                  # USD/t
    m.a = pyo.Param(m.S, m.P, initialize={(s, p): float(use0.loc[s, p])
                                          for s in OPS for p in GRADES})  # h/t
    m.b = pyo.Param(m.S, initialize=bb.to_dict())                       # h/week
    m.x = pyo.Var(m.P, domain=pyo.NonNegativeReals)                     # t/week
    m.op = pyo.Constraint(m.S, rule=lambda m, s:                        # R1, capacity
                          sum(m.a[s, p] * m.x[p] for p in m.P) <= m.b[s])
    m.z = pyo.Objective(expr=sum(m.c[p] * m.x[p] for p in m.P), sense=pyo.maximize)
    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    return m


lp = pick_solver("lp")
m0 = build_warmup()
r0 = lp.solve(m0)
print("\ntermination condition:", r0.solver.termination_condition)
assert r0.solver.termination_condition == pyo.TerminationCondition.optimal, \
    "warm-up product-mix LP did not solve to optimality"

x0 = np.array([pyo.value(m0.x[p]) for p in GRADES])
z0 = float(pyo.value(m0.z))
used0 = use0.values @ x0
dual0 = np.abs([m0.dual[m0.op[s]] for s in OPS])

print(f"\noptimal plan: " + ", ".join(f"{p} = {v:.4g} t/week" for p, v in zip(GRADES, x0)))
print(f"optimal contribution margin: {z0:,.2f} USD/week\n")
slack0 = np.where(np.abs(avail0.values - used0) < 1e-9, 0.0, avail0.values - used0)
print(pd.DataFrame({"used_h": used0, "available_h": avail0.values, "slack_h": slack0,
                    "binding": np.isclose(slack0, 0.0, atol=1e-7),
                    "shadow_price_USD_per_h": dual0}, index=OPS).round(4).to_string())

In [ ]:
# --- Stage 5 continued: two independent checks before the answer is believed --
# Check A: vertex enumeration in NumPy, the Week 1 method, on the same data.
A0, b0, c0 = use0.values, avail0.values, margin0.values
bnd_A   = np.vstack([A0, np.eye(2)])
bnd_rhs = np.concatenate([b0, np.zeros(2)])
best_x, best_z = None, -np.inf
for i in range(5):
    for j in range(i + 1, 5):
        M = bnd_A[[i, j], :]
        if abs(np.linalg.det(M)) < 1e-9:
            continue
        v = np.linalg.solve(M, bnd_rhs[[i, j]])
        if np.all(A0 @ v <= b0 + 1e-7) and np.all(v >= -1e-7) and c0 @ v > best_z:
            best_x, best_z = v, float(c0 @ v)
print(f"vertex enumeration : x = ({best_x[0]:.4g}, {best_x[1]:.4g}), z = {best_z:,.2f} USD/week")
print(f"Pyomo              : x = ({x0[0]:.4g}, {x0[1]:.4g}), z = {z0:,.2f} USD/week")
assert np.allclose(best_x, x0, atol=1e-7) and abs(best_z - z0) < 1e-7, \
    "vertex enumeration and Pyomo disagree"

# Check B: every shadow price against a finite difference, one extra hour at a time.
print("\nfinite-difference check of each shadow price")
for k, s in enumerate(OPS):
    bumped = avail0.copy(); bumped[s] += 1.0
    mm = build_warmup(avail_h=bumped)
    rr = lp.solve(mm)
    assert rr.solver.termination_condition == pyo.TerminationCondition.optimal
    obs = float(pyo.value(mm.z)) - z0
    print(f"  {s:<13s} dual {dual0[k]:7.2f}   observed {obs:7.2f}   "
          f"{'match' if abs(obs - dual0[k]) < 1e-6 else 'MISMATCH'}")
    assert abs(obs - dual0[k]) < 1e-6, f"shadow price of {s} is not reproduced"
print("\nBoth checks pass, so the stage 5 numbers may be quoted.")

### Stage 5: interpretation

The plan finishes 24 t of Standard and 20 t of HighPurity, for 28,280.00 USD of contribution margin in the week. Purification and filling are both fully used and mixing has 12 h of slack, so the shop is limited by purification and filling, not by mixing capacity. The two binding operations carry shadow prices of 90.00 USD per purification hour and 430.00 USD per filling hour, and both are reproduced exactly by re-solving with one extra hour. Buying mixing hours is worth nothing at this data.

The engineering reading is that an overtime shift should be bought on filling first, not on the operation that looks busiest by absolute hours. Mixing consumes 108 of the 120 allocated hours, more than either other operation in absolute terms, and is worth zero at the margin.

One limitation matters for the rest of the notebook. This model plans a single week in isolation. It has no memory: it cannot build stock now for a peak later, and it cannot express that a shortfall this week must be delivered next week. Adding a time index and one new constraint family fixes that, and is the subject of Section 3.


## 2. Four reusable modeling patterns

Linear models in engineering are not invented from scratch each time. Almost every linear model in this course is assembled from four patterns. Naming them is worth doing, because stage 3 of the workflow then becomes a matter of recognizing which patterns apply rather than deriving algebra from nothing.

**Pattern 1: the index set.** Give every dimension of the problem a named set, and index variables, parameters and constraints over it. `x_p` becomes `x_{p,t}` when time is added, and the constraint that was written once is now written once per period. The gain is that adding a product, a period or a site is a change to the *data*, not to the model. This is why the Pyomo `Set` component exists and why data is kept in a DataFrame rather than typed into a rule.

**Pattern 2: the balance equation.** Any conserved quantity satisfies

    accumulation = inflow - outflow

around any boundary you choose to draw. Draw the boundary around a period and it is an inventory balance. Draw it around a location and it is a node balance. Draw it around a piece of equipment and it is a material balance on a unit. The algebra is the same and only the index changes. Balance equations are equalities, they are the backbone of the model, and a missing one is the most expensive kind of error because the model will still solve.

**Pattern 3: the capacity constraint.** Any limited resource gives one inequality per resource, and per period if the resource is renewed each period:

    sum over activities of (use per unit of activity) * (level of activity) <= availability

Capacity constraints are where the units check pays. The left side must carry the resource's own unit, hours or tonnes or cubic meters, not the unit of the activity.

**Pattern 4: the linking constraint.** A linking, or coupling, constraint is any constraint whose index reaches across more than one otherwise independent block of the model. Remove all linking constraints and the model falls apart into small independent problems. Two kinds appear in this notebook:

- *temporal linking*: the inventory balance carries stock from period `t-1` into period `t`, so the periods cannot be planned one at a time;
- *resource linking*: one shared reactor is used by both products, so the products cannot be planned one at a time.

The linking constraints are where the value of optimization lives. Section 4 measures the value of temporal linking by comparing the optimal plan with a myopic period-by-period rule, and Section 6 measures the value of resource linking by comparing the coupled model with the two products solved separately.


In [ ]:
# --- The pattern catalog, and which patterns each model in this notebook uses
patterns = pd.DataFrame([
    ["1 index set",
     "i in I; x_i, and x_{i,t} when a dimension is added",
     "names each dimension once so that growth is a data change",
     "pyo.Set(initialize=...)",
     "every week of the course"],
    ["2 balance equation",
     "accumulation = inflow - outflow, e.g. I_{t-1} + x_t - d_t = I_t",
     "conserves a quantity across a chosen boundary",
     "equality Constraint indexed over the boundary",
     "node balance W3, component balance W4, unit balance W3 and W4"],
    ["3 capacity constraint",
     "sum_i a_ri x_i <= b_r",
     "limits activity on a resource that is in short supply",
     "inequality Constraint indexed over resources",
     "arc capacity W3, pool capacity W4, unit capacity W5"],
    ["4 linking constraint",
     "any constraint whose index spans more than one block",
     "destroys separability and creates the real decision",
     "Constraint indexed over the shared dimension only",
     "transshipment nodes W3, pooling W4, big-M links W5"],
], columns=["pattern", "algebraic template", "what it does", "Pyomo form", "reappears in"])
for _, row in patterns.iterrows():
    print(f"{row['pattern'].upper()}")
    print(f"   template     : {row['algebraic template']}")
    print(f"   what it does : {row['what it does']}")
    print(f"   Pyomo form   : {row['Pyomo form']}")
    print(f"   reappears in : {row['reappears in']}")

# Which patterns each model in this notebook uses.
print("\nPattern usage in this notebook")
print(f"  Section 1 warm-up       : {len(m0.op)} capacity constraints over |S| = {len(m0.S)} "
      f"operations; no balance, no linking")
print("  Section 3 single product: one balance equation per period (pattern 2, and pattern 4")
print("                            because it links t-1 to t) plus one capacity constraint")
print("                            per period (pattern 3)")
print("  Section 5 two products  : balance and line constraints per (product, period), plus")
print("                            one shared reactor constraint per period (pattern 4,")
print("                            resource linking)")
print("\nThe constraint counts printed when those models are built confirm these structures.")

## 3. Problem statement, single product

A specialty chemical plant produces an electrolyte additive used in zinc-air battery assembly. Planning covers
a horizon of six monthly periods, `t = 1, ..., 6`.

Each period the plant may produce up to a line capacity. Output that is not sold immediately is carried as
inventory at a holding cost. Demand that cannot be met in the period it appears is not lost: it is backlogged
and delivered later, at a penalty that represents expedited freight and contractual damages. Production cost
per tonne varies by period because the plant buys electricity on a seasonal contract.

### Sets and parameters

| Symbol | Meaning | Units |
|---|---|---|
| `t in T` | planning period | month |
| `d_t` | demand | t/month |
| `C_t` | production capacity | t/month |
| `c_t` | unit production cost | USD/t |
| `h` | inventory holding cost | USD/t/month |
| `b` | backlog penalty | USD/t/month |
| `I_0` | initial inventory | t |

### Decision variables

- `x_t >= 0` production in period `t`, tonnes
- `I_t >= 0` inventory carried out of period `t`, tonnes
- `B_t >= 0` backlog carried out of period `t`, tonnes

### Model

Minimize total cost

    min  sum_t ( c_t x_t + h I_t + b B_t )

subject to the inventory balance for every period

    I_{t-1} - B_{t-1} + x_t - d_t = I_t - B_t        for all t in T

the capacity limit

    x_t <= C_t                                       for all t in T

the horizon-end service condition

    B_{|T|} = 0

and non-negativity `x_t, I_t, B_t >= 0`, with `I_0` given and `B_0 = 0`.

The single quantity `I_t - B_t` is the net stock position. Splitting it into two non-negative variables keeps
the model linear while allowing holding and shortage to be priced differently. Because `h > 0` and `b > 0`
and the objective is a minimization, no optimal solution ever has `I_t > 0` and `B_t > 0` at the same time:
carrying stock and owing stock simultaneously would cost money and change nothing. The two variables are
therefore complementary without any binary variable being needed.

The horizon-end condition `B_6 = 0` is what makes the plan honest. Without it the model would simply push
all difficult demand past the end of the horizon.

In [ ]:
# --- Data, single product ----------------------------------------------------
T = list(range(1, 7))

data1 = pd.DataFrame({
    "period":        T,
    "demand_t":      [120, 150, 180, 210, 160, 130],
    "capacity_t":    [170, 170, 170, 170, 170, 170],
    "prod_cost_usd": [42.0, 42.0, 45.0, 48.0, 46.0, 43.0],
}).set_index("period")

HOLD_COST = 6.0     # USD per tonne per month
BACK_COST = 45.0    # USD per tonne per month
I_INIT    = 20.0    # tonnes on hand at the start of period 1

print(data1)
print()
print(f"Total demand over the horizon : {data1.demand_t.sum():>6.0f} t")
print(f"Total capacity over the horizon: {data1.capacity_t.sum():>6.0f} t")
print(f"Slack including opening stock  : "
      f"{data1.capacity_t.sum() + I_INIT - data1.demand_t.sum():>6.0f} t")

The horizon is feasible in aggregate, but not period by period. Demand in period 4 is 210 t against a
capacity of 170 t, so the plan must either build stock in advance or accept a backlog. That single tension is
the whole point of a multiperiod model.

In [ ]:
# --- Pyomo model, single product ---------------------------------------------
def build_single(df, hold=HOLD_COST, back=BACK_COST, I0=I_INIT):
    m = pyo.ConcreteModel(name="single_product_multiperiod")

    m.T = pyo.Set(initialize=list(df.index), ordered=True)

    m.d = pyo.Param(m.T, initialize=df.demand_t.to_dict())
    m.C = pyo.Param(m.T, initialize=df.capacity_t.to_dict())
    m.c = pyo.Param(m.T, initialize=df.prod_cost_usd.to_dict())
    m.h = pyo.Param(initialize=hold)
    m.b = pyo.Param(initialize=back)
    m.I0 = pyo.Param(initialize=I0)

    m.x = pyo.Var(m.T, domain=pyo.NonNegativeReals)   # production
    m.I = pyo.Var(m.T, domain=pyo.NonNegativeReals)   # inventory
    m.B = pyo.Var(m.T, domain=pyo.NonNegativeReals)   # backlog

    def balance_rule(m, t):
        prev_I = m.I0 if t == m.T.first() else m.I[m.T.prev(t)]
        prev_B = 0.0  if t == m.T.first() else m.B[m.T.prev(t)]
        return prev_I - prev_B + m.x[t] - m.d[t] == m.I[t] - m.B[t]
    m.balance = pyo.Constraint(m.T, rule=balance_rule)

    m.capacity = pyo.Constraint(m.T, rule=lambda m, t: m.x[t] <= m.C[t])
    m.no_final_backlog = pyo.Constraint(expr=m.B[m.T.last()] == 0)

    m.cost = pyo.Objective(
        expr=sum(m.c[t]*m.x[t] + m.h*m.I[t] + m.b*m.B[t] for t in m.T),
        sense=pyo.minimize)

    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    return m

m1 = build_single(data1)
print(f"variables  : {m1.nvariables()}")
print(f"constraints: {m1.nconstraints()}")

In [ ]:
# --- Solve and check status --------------------------------------------------
lp = pick_solver("lp")
res1 = lp.solve(m1)

print(res1.solver.termination_condition)
assert res1.solver.termination_condition == pyo.TerminationCondition.optimal, \
    "single-product LP did not solve to optimality"

cost1 = pyo.value(m1.cost)
print(f"Optimal total cost: {cost1:,.2f} USD")

---

### The same model in the spreadsheet

The companion workbook for this week is `excel/W02_multiperiod_planning_STUDENT.xlsx`, with the
completed version in `excel/W02_multiperiod_planning_SOLUTION.xlsx`. Per the tool allocation
table of the course specification, Week 2 is an **OpenSolver week**: the spreadsheet is the primary tool
because periods laid across columns read naturally, and Pyomo is the comparison made at the end. On the
`Model` sheet the six periods occupy columns `B` to `G`, so every indexed family of the algebra becomes
a row that is copied across those six columns.

| Algebraic symbol | Spreadsheet range or layout | Pyomo component | Note |
|---|---|---|---|
| `t in T`, planning period | header row `B3:G3`, one column per period | `m.T = pyo.Set(initialize=..., ordered=True)` | Order matters in this model: `m.T.prev(t)` is the column immediately to the left. |
| `d_t`, demand, t/month | data row `B15:G15`, blue text | `m.d = pyo.Param(m.T, initialize=...)` | Data rows sit below the model block so the solved plan stays visible on one screen. |
| `C_t`, capacity, t/month | data row `B16:G16`, blue text, which is also the `RHS` row of the capacity family | `m.C = pyo.Param(m.T, initialize=...)` | A parameter row and a right-hand side row can be the same row when the constraint is a simple bound. |
| `c_t`, unit production cost | objective coefficient row `B9:G9`, blue text | `m.c = pyo.Param(m.T, initialize=...)` | The only genuinely period-dependent cost; `h` and `b` are scalars. |
| `h`, `b`, holding and backlog cost | single labeled cells `B10` and `B11`, blue text | `m.h`, `m.b`, scalar `pyo.Param` | A scalar parameter occupies a cell, not a row. Referencing it absolutely, `$B$10`, is what keeps the objective copyable. |
| `I_0`, opening inventory | single cell `A6`, sitting immediately left of the inventory `Solution` row | `m.I0 = pyo.Param(initialize=I_INIT)` | Putting period zero in column `A` is the trick that makes one balance formula copy across all six periods. |
| `x_t >= 0`, production | `Solution` row `B5:G5`, green fill | `m.x = pyo.Var(m.T, domain=pyo.NonNegativeReals)` | Changing cells. The three green rows together are the changing-cell range `$B$5:$G$7`. |
| `I_t >= 0`, inventory | `Solution` row `B6:G6`, green fill | `m.I = pyo.Var(m.T, domain=pyo.NonNegativeReals)` | Splitting net stock into `I` and `B` costs one extra row in the sheet and one extra `Var` in Pyomo. |
| `B_t >= 0`, backlog | `Solution` row `B7:G7`, green fill | `m.B = pyo.Var(m.T, domain=pyo.NonNegativeReals)` | No binary is needed to keep `I_t` and `B_t` from being positive together; the cost structure does it. |
| `I_{t-1} - B_{t-1} + x_t - d_t = I_t - B_t` | `VALUE` row `B18:G18`, each cell `=A6-A7+B5-B15-B6+B7` copied right; relation `=` labeled in `H18`; `RHS` row `B19:G19` of zeros | `m.balance = pyo.Constraint(m.T, rule=balance_rule)` | Note the transposition: with periods across columns, the Week 1 `VALUE`, relation and `RHS` **columns** become a `VALUE` row, one relation label and an `RHS` row. |
| `x_t <= C_t` | dialog entry `$B$5:$G$5 <= $B$16:$G$16`, no `SUMPRODUCT` required | `m.capacity = pyo.Constraint(m.T, rule=...)` | A range-against-range entry is the spreadsheet form of an indexed constraint. |
| `B_{|T|} = 0` | single cell entry `$G$7 = 0` | `m.no_final_backlog = pyo.Constraint(expr=...)` | The only scalar constraint in the model, and the one that makes the plan honest. |
| `min sum_t (c_t x_t + h I_t + b B_t)` | objective cell `C13`, `=SUMPRODUCT(B9:G9,B5:G5)+$B$10*SUM(B6:G6)+$B$11*SUM(B7:G7)`, entered with To: Min | `m.cost = pyo.Objective(expr=..., sense=pyo.minimize)` | Three terms in one cell, matching the three terms of the algebraic sum. |
| `p in P` and `sum_p rho_p x_pt <= R_t` | the whole block copied to rows 25 to 40 with identical column geometry, plus a reactor `VALUE` row `B42:G42` of `SUMPRODUCT`s over both production rows, a relation label, and an `RHS` row `B43:G43` | `m.P = pyo.Set(...)`, `m.reactor = pyo.Constraint(m.T, rule=...)` | A second product is a second block of the same shape. In Pyomo it is one extra index on components that already exist. |

**Where the spreadsheet stops working.** At the size used here the sheet is comfortable: one product,
six periods, 18 changing cells, 13 constraint cells, all visible at once. Now take the identical model
to the planning cycle a real plant runs, weekly buckets over a year with three products. The `Solution`
block becomes 3 rows by 52 columns by 3 products, that is 468 changing cells; the balance family becomes
156 formulas, each a copy whose relative and absolute references must be correct in every one of 52
columns, and a single mis-anchored `$` in a single column produces a plan the sheet reports as feasible
and the plant cannot execute. The shared reactor adds 52 more rows, each with a dual that has to be read
out of a sensitivity report by hand. The debottlenecking scan behind Figure 3 sweeps reactor hours from
320 to 384 in steps of 2, which is 33 re-solves: in the sheet that is 33 rounds of edit a cell, reopen
the dialog, click Solve, copy the objective out, with no record of what was clicked. In the model above,
going from 6 periods to 52 is one change to the index of `data1`, adding a product is one entry in `P`,
and the scan is a `for` loop that leaves an auditable array behind.

---

In [ ]:
# --- Results table -----------------------------------------------------------
plan1 = pd.DataFrame({
    "demand":     [pyo.value(m1.d[t]) for t in T],
    "capacity":   [pyo.value(m1.C[t]) for t in T],
    "production": [pyo.value(m1.x[t]) for t in T],
    "inventory":  [pyo.value(m1.I[t]) for t in T],
    "backlog":    [pyo.value(m1.B[t]) for t in T],
    "unit_cost":  [pyo.value(m1.c[t]) for t in T],
}, index=pd.Index(T, name="period")).round(3)

plan1["cap_dual"] = [round(m1.dual[m1.capacity[t]], 4) for t in T]
print(plan1)
print()
print(f"production cost : {sum(pyo.value(m1.c[t]*m1.x[t]) for t in T):>10,.2f} USD")
print(f"holding cost    : {sum(HOLD_COST*pyo.value(m1.I[t]) for t in T):>10,.2f} USD")
print(f"backlog cost    : {sum(BACK_COST*pyo.value(m1.B[t]) for t in T):>10,.2f} USD")
print(f"total           : {cost1:>10,.2f} USD")

In [ ]:
# --- Benchmark: a myopic period-by-period rule -------------------------------
# Produce only what is needed now, up to capacity. No look-ahead.
net, myopic_cost, myopic_rows = I_INIT, 0.0, []
for t in T:
    need = max(data1.demand_t[t] - net, 0.0)
    x = min(need, data1.capacity_t[t])
    net = net + x - data1.demand_t[t]
    I, B = max(net, 0.0), max(-net, 0.0)
    myopic_cost += data1.prod_cost_usd[t]*x + HOLD_COST*I + BACK_COST*B
    myopic_rows.append((x, I, B))

myopic = pd.DataFrame(myopic_rows, columns=["production", "inventory", "backlog"],
                      index=pd.Index(T, name="period"))
print(myopic)
print(f"\nmyopic cost   : {myopic_cost:>10,.2f} USD")
print(f"optimal cost  : {cost1:>10,.2f} USD")
print(f"value of look-ahead: {myopic_cost - cost1:,.2f} USD "
      f"({100*(myopic_cost - cost1)/myopic_cost:.1f} % of the myopic cost)")

In [ ]:
# --- Figure 1: the optimal plan ----------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.4), constrained_layout=True)

ax = tidy(axes[0])
w = 0.38
ax.bar([t - w/2 for t in T], plan1.demand, width=w, color=PALETTE[0], label="demand")
ax.bar([t + w/2 for t in T], plan1.production, width=w, color=PALETTE[1], label="production")
ax.plot(T, plan1.capacity, color=GRAPHITE, ls="--", lw=1.5, label="capacity")
ax.set_xlabel("period"); ax.set_ylabel("tonnes"); ax.set_xticks(T)
ax.set_title("Production against demand")
ax.legend(loc="upper left", fontsize=8)

ax = tidy(axes[1])
ax.fill_between(T, 0, plan1.inventory, color=PALETTE[7], alpha=0.85, step=None,
                label="inventory (optimal)")
ax.plot(T, plan1.inventory, color=PALETTE[0], marker="o", label="_nolegend_")
ax.plot(T, myopic.inventory - myopic.backlog, color=PALETTE[1], marker="s", ls="--",
        label="net stock (myopic)")
ax.axhline(0, color=GRAPHITE, lw=1.0)
ax.set_xlabel("period"); ax.set_ylabel("net stock, tonnes"); ax.set_xticks(T)
ax.set_title("Optimal pre-build against myopic backlog")
ax.legend(loc="lower left", fontsize=8)

fig.suptitle("Week 2, Figure 1: single-product multiperiod plan", color=INK, fontsize=10)
plt.show()

## 4. Interpretation of the single-product plan

The optimal cost is 42,080.00 USD, made up of 41,360.00 USD of production, 720.00 USD of holding, and no
backlog cost at all. The myopic rule costs 45,940.00 USD, so look-ahead is worth 3,860.00 USD, or 8.4 percent
of the myopic cost.

**Why the plan pre-builds.** Periods 3 and 4 together demand 390 t against a combined capacity of 340 t, a
shortfall of 50 t that physically must be manufactured earlier. The optimal inventory at the end of period 2
is exactly 50 t, which is the shortfall and nothing more. The pre-build is therefore driven by capacity, not
by the price of electricity. Price arbitrage on its own would not justify it: producing in period 2 at
42 USD/t and holding for two months costs 42 + 2(6) = 54 USD/t, which is worse than simply producing in
period 4 at 48 USD/t. The plant builds stock in spite of the price signal because it has no alternative.

**Why the plan carries no backlog.** A backlog is charged again in every period until it is cleared. The
myopic rule falls behind in period 3 and pays the 45 USD/t/month penalty in periods 3, 4 and 5 on a stock
that peaks at 50 t, which is why its cost is so much higher even though it produces the same total tonnage.

**Reading the capacity duals.** The `cap_dual` column is 0, -6, -9, -12, 0, 0 for periods 1 to 6. Pyomo
returns a negative dual for a `<=` constraint in a minimization, so the magnitude is the cost reduction from
one extra tonne of capacity in that period. The value 12 USD/t in period 4 is not arbitrary. The marginal
tonne delivered in period 4 currently originates in period 1, the only early period with spare capacity, and
is held for three months: 42 + 3(6) = 60 USD/t against 48 USD/t if it could be made in period 4, a difference
of 12 USD/t. The same accounting gives 9 for period 3 and 6 for period 2. Periods 1, 5 and 6 have slack
capacity and a dual of zero, so extra capacity there is worthless.

This is the practical output of the model. If an operations manager asks where to spend a debottlenecking
budget, the answer is period 4 first, at 12 USD per tonne of extra capacity, and nowhere near periods 1, 5
or 6.

## 5. Adding a second product and a shared reactor

The plant now also produces a cathode binder. Each product has its own finishing line, so the line capacities
stay separate, but both products are made in the same reactor train. Reactor time is the shared resource.

### What changes in the formulation

Every variable and every balance gains a product index `p`:

    x_{p,t}, I_{p,t}, B_{p,t}

The inventory balance and the line capacity are written once per product and are otherwise unchanged:

    I_{p,t-1} - B_{p,t-1} + x_{p,t} - d_{p,t} = I_{p,t} - B_{p,t}     for all p, t
    x_{p,t} <= C_{p,t}                                                for all p, t

One genuinely new constraint appears. It is indexed over time only, and it is the only place where the two
products meet:

    sum_p rho_p x_{p,t} <= R_t                                        for all t

where `rho_p` is reactor hours per tonne of product `p` and `R_t` is reactor hours available in period `t`.

The objective is the sum over products of the same cost terms as before:

    min  sum_p sum_t ( c_{p,t} x_{p,t} + h_p I_{p,t} + b_p B_{p,t} )

This is the structural lesson of the week. Without the reactor constraint the problem is block-diagonal: it
separates exactly into one independent LP per product, and solving them jointly gains nothing. The shared
reactor is a *coupling constraint*. It links the blocks, destroys separability, and creates a genuine
allocation decision: in a period where reactor time is short, which product gets it.

In [ ]:
# --- Data, two products ------------------------------------------------------
P = ["Additive", "Binder"]

demand2 = pd.DataFrame(
    {"Additive": [120, 150, 180, 210, 160, 130],
     "Binder":   [ 90, 110, 130, 140, 120, 100]},
    index=pd.Index(T, name="period"))

cost2 = pd.DataFrame(
    {"Additive": [42.0, 42.0, 45.0, 48.0, 46.0, 43.0],
     "Binder":   [58.0, 58.0, 61.0, 64.0, 62.0, 59.0]},
    index=pd.Index(T, name="period"))

line_cap  = {"Additive": 170.0, "Binder": 150.0}   # t/month, dedicated finishing lines
hold2     = {"Additive":   6.0, "Binder":   8.0}   # USD/t/month
back2     = {"Additive":  45.0, "Binder":  60.0}   # USD/t/month
rho       = {"Additive":   1.0, "Binder":   1.4}   # reactor hours per tonne
I_init2   = {"Additive":  20.0, "Binder":  10.0}   # tonnes
REACTOR_H = 330.0                                  # reactor hours available per month

print("demand (t/month)\n", demand2, "\n")
print("reactor hours needed if each product is made exactly to demand:")
need = (demand2["Additive"]*rho["Additive"] + demand2["Binder"]*rho["Binder"]).round(1)
print(pd.DataFrame({"hours_needed": need, "hours_available": REACTOR_H,
                    "surplus": (REACTOR_H - need).round(1)}))

The just-in-time reactor requirement exceeds the 330 available hours in the middle periods, so the
reactor constraint will bind and the plan must shift work into the early periods.

In [ ]:
# --- Pyomo model, two products with a shared reactor -------------------------
def build_multi(reactor_hours=REACTOR_H, couple=True):
    m = pyo.ConcreteModel(name="multiproduct_multiperiod")

    m.P = pyo.Set(initialize=P)
    m.T = pyo.Set(initialize=T, ordered=True)

    m.d   = pyo.Param(m.P, m.T, initialize={(p, t): demand2.loc[t, p] for p in P for t in T})
    m.c   = pyo.Param(m.P, m.T, initialize={(p, t): cost2.loc[t, p]   for p in P for t in T})
    m.C   = pyo.Param(m.P, initialize=line_cap)
    m.h   = pyo.Param(m.P, initialize=hold2)
    m.b   = pyo.Param(m.P, initialize=back2)
    m.rho = pyo.Param(m.P, initialize=rho)
    m.I0  = pyo.Param(m.P, initialize=I_init2)
    m.R   = pyo.Param(m.T, initialize={t: reactor_hours for t in T}, mutable=True)

    m.x = pyo.Var(m.P, m.T, domain=pyo.NonNegativeReals)
    m.I = pyo.Var(m.P, m.T, domain=pyo.NonNegativeReals)
    m.B = pyo.Var(m.P, m.T, domain=pyo.NonNegativeReals)

    def balance_rule(m, p, t):
        prev_I = m.I0[p] if t == m.T.first() else m.I[p, m.T.prev(t)]
        prev_B = 0.0     if t == m.T.first() else m.B[p, m.T.prev(t)]
        return prev_I - prev_B + m.x[p, t] - m.d[p, t] == m.I[p, t] - m.B[p, t]
    m.balance = pyo.Constraint(m.P, m.T, rule=balance_rule)

    m.line = pyo.Constraint(m.P, m.T, rule=lambda m, p, t: m.x[p, t] <= m.C[p])

    if couple:
        m.reactor = pyo.Constraint(
            m.T, rule=lambda m, t: sum(m.rho[p]*m.x[p, t] for p in m.P) <= m.R[t])

    m.no_final_backlog = pyo.Constraint(m.P, rule=lambda m, p: m.B[p, m.T.last()] == 0)

    m.cost = pyo.Objective(
        expr=sum(m.c[p, t]*m.x[p, t] + m.h[p]*m.I[p, t] + m.b[p]*m.B[p, t]
                 for p in m.P for t in m.T),
        sense=pyo.minimize)

    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    return m

m2 = build_multi()
res2 = lp.solve(m2)
print(res2.solver.termination_condition)
assert res2.solver.termination_condition == pyo.TerminationCondition.optimal, \
    "coupled two-product LP did not solve to optimality"
cost2_val = pyo.value(m2.cost)
print(f"Coupled optimal cost: {cost2_val:,.2f} USD")

In [ ]:
# --- Results table, two products ---------------------------------------------
rows = []
for p in P:
    for t in T:
        rows.append({"product": p, "period": t,
                     "demand":     pyo.value(m2.d[p, t]),
                     "production": pyo.value(m2.x[p, t]),
                     "inventory":  pyo.value(m2.I[p, t]),
                     "backlog":    pyo.value(m2.B[p, t]),
                     "reactor_h":  pyo.value(m2.rho[p]*m2.x[p, t])})
plan2 = pd.DataFrame(rows).round(3)
print(plan2.pivot(index="period", columns="product",
                  values=["production", "inventory", "backlog"]).round(2))

react = pd.DataFrame({
    "hours_used":  [sum(pyo.value(m2.rho[p]*m2.x[p, t]) for p in P) for t in T],
    "hours_avail": [pyo.value(m2.R[t]) for t in T],
    "dual":        [m2.dual[m2.reactor[t]] for t in T],
}, index=pd.Index(T, name="period")).round(4)
react["binding"] = react.hours_used > react.hours_avail - 1e-6
print("\nShared reactor:")
print(react)

In [ ]:
# --- Cost of coupling: solve the two products separately ---------------------
m2_free = build_multi(couple=False)
res2f = lp.solve(m2_free)
assert res2f.solver.termination_condition == pyo.TerminationCondition.optimal
cost_uncoupled = pyo.value(m2_free.cost)

peak = max(sum(pyo.value(m2_free.rho[p]*m2_free.x[p, t]) for p in P) for t in T)

print(f"uncoupled cost (no reactor limit) : {cost_uncoupled:>12,.2f} USD")
print(f"coupled cost   (reactor <= {REACTOR_H:.0f} h) : {cost2_val:>12,.2f} USD")
print(f"cost of the shared reactor        : {cost2_val - cost_uncoupled:>12,.2f} USD")
print(f"peak reactor hours the uncoupled plan would need: {peak:.1f} h")

In [ ]:
# --- Marginal value of reactor time: dual against a finite-difference check ---
t_star = int(react.dual.abs().idxmax())
print(f"Most valuable period for extra reactor time: t = {t_star}, "
      f"dual = {react.dual[t_star]:.4f} USD per hour\n")

m_plus = build_multi()
m_plus.R[t_star] = REACTOR_H + 1.0
r_plus = lp.solve(m_plus)
assert r_plus.solver.termination_condition == pyo.TerminationCondition.optimal,     "perturbed model did not solve to optimality"
delta = pyo.value(m_plus.cost) - cost2_val
print(f"cost with one extra reactor hour in period {t_star}: {pyo.value(m_plus.cost):,.4f} USD")
print(f"finite-difference change                   : {delta:+.4f} USD")
print(f"LP dual                                    : {react.dual[t_star]:+.4f} USD")
assert abs(delta - react.dual[t_star]) < 1e-4, "dual and finite difference disagree"
print("\nThe dual reproduces the finite difference exactly, as LP theory requires "
      "for a step small enough to keep the same optimal basis.")

In [ ]:
# --- Figure 2: reactor allocation and its shadow price -----------------------
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.4), constrained_layout=True)

ax = tidy(axes[0])
hrs = plan2.pivot(index="period", columns="product", values="reactor_h")
ax.bar(T, hrs["Additive"], color=PALETTE[0], label="Additive")
ax.bar(T, hrs["Binder"], bottom=hrs["Additive"], color=PALETTE[1], label="Binder")
ax.plot(T, react.hours_avail, color=GRAPHITE, ls="--", lw=1.5, label="reactor limit")
ax.set_xlabel("period"); ax.set_ylabel("reactor hours"); ax.set_xticks(T)
ax.set_title("Reactor time allocated by product")
ax.legend(loc="lower center", ncol=3, fontsize=8)

ax = tidy(axes[1])
mag = (-react.dual).values          # cost reduction per extra hour
ax.bar(T, mag, color=[PALETTE[1] if v > 1e-6 else MIST for v in mag])
ax.set_xlabel("period"); ax.set_ylabel("USD per extra reactor hour"); ax.set_xticks(T)
ax.set_title("Shadow price of reactor time")
for t, v in zip(T, mag):
    if v > 1e-6:
        ax.text(t, v + 0.15, f"{v:.2f}", ha="center", fontsize=7.5, color=INK)

fig.suptitle("Week 2, Figure 2: the shared reactor couples the two products",
             color=INK, fontsize=10)
plt.show()

In [ ]:
# --- Figure 3: how the plan responds to reactor debottlenecking --------------
scan = np.arange(320.0, 386.0, 2.0)
costs, peak_backlog = [], []
for R in scan:
    mm = build_multi(reactor_hours=float(R))
    r = lp.solve(mm)
    assert r.solver.termination_condition == pyo.TerminationCondition.optimal
    costs.append(pyo.value(mm.cost))
    peak_backlog.append(max(pyo.value(mm.B[p, t]) for p in P for t in T))
costs = np.array(costs)

fig, ax = plt.subplots(figsize=(5.4, 3.4), constrained_layout=True)
tidy(ax)
ax.plot(scan, costs, color=PALETTE[0], marker="o", ms=4, label="optimal total cost")
ax.axhline(cost_uncoupled, color=PALETTE[1], ls="--", lw=1.5,
           label="uncoupled lower bound")
ax.axvline(REACTOR_H, color=GRAPHITE, ls=":", lw=1.3)
ax.annotate(f"base case\n{REACTOR_H:.0f} h", xy=(REACTOR_H, costs.max()),
            xytext=(REACTOR_H + 2, costs.max()), fontsize=8, color=INK, va="top")
ax.set_xlabel("reactor hours available per period")
ax.set_ylabel("optimal total cost, USD")
ax.set_title("Week 2, Figure 3: value of debottlenecking the reactor", fontsize=10)
ax.legend(fontsize=8)
plt.show()

flat = np.isclose(costs, cost_uncoupled, atol=1e-6)
if flat.any():
    print(f"cost reaches the uncoupled lower bound at {scan[np.argmax(flat)]:.0f} reactor hours "
          f"per period, which is the first grid point at or above the uncoupled peak "
          f"requirement of {peak:.1f} h")
else:
    print("the reactor still binds across the whole scan")

## 6. Interpretation of the coupled model

Three points are worth extracting from the output above.

**The coupling constraint is what makes this a single LP.** Solving the two products separately gives
83,330.00 USD, but that plan needs 366.0 reactor hours in period 4 and only 330 are available, so it is not a
plan at all. The coupled optimum is 83,930.86 USD. The difference, 600.86 USD over the six-month horizon, is
what the plant pays for owning one reactor train instead of two, and it is the correct figure to weigh
against the annualized cost of extra reactor capacity. Notice how small it is: sharing is cheap here because
the two demand peaks can be smoothed by inventory.

**The shadow price is periodic, not constant.** The reactor duals are 0, -5.7143, -9.2857, -12.2857, 0, 0 for
periods 1 to 6. Only periods 2, 3 and 4 bind. An extra hour in period 4 saves 12.2857 USD, an extra hour in
period 1 or 5 saves nothing. The finite-difference check in Section 5 confirms this: re-solving with 331
hours in period 4 gives 83,918.5714 USD, exactly 12.2857 USD below the base case. That agreement is not a
coincidence, it is the definition of the dual, and it holds for any perturbation small enough to leave the
optimal basis unchanged. A capital request should therefore always name the period, not just the resource.

**The debottlenecking curve is piecewise linear and eventually flat.** Figure 3 shows total cost falling as
reactor hours are added and then flattening. It reaches the uncoupled lower bound of 83,330.00 USD at 366
hours per period, which is precisely the peak requirement of the uncoupled plan. Beyond that the reactor is
no longer limiting anywhere in the horizon and further investment returns exactly zero. Any economic
evaluation that extrapolates the initial slope past that kink will overstate the benefit.

## 7. Exercises

**Exercise 1 (introductory, warm-up model).** Restriction R3 of the Section 1 specification assumes that everything finished is sold. Remove that assumption: the HighPurity grade can be sold at most 15 t in the week. Add the bound to the warm-up model, re-solve, and report the new plan, the new margin and the new shadow prices of all three operations. State in one sentence which stage of the workflow the error would have been caught at if the sales limit had existed all along and had been omitted.

**Exercise 2 (introductory).** Change the backlog penalty for the Additive from 45 to 15 USD/t/month and
re-solve the single-product model. Report the new plan, the new cost, and the periods in which a backlog now
appears. Explain in one sentence why the plan changes at that penalty level, using the holding cost and the
period-to-period production cost differences.

**Exercise 3 (introductory).** Add a maximum warehouse capacity of 60 t on the total inventory in any period
of the two-product model, that is `sum_p I_{p,t} <= 60`. Re-solve, report the new cost, and report the dual
of the warehouse constraint in every period. Which is worth more at the margin in period 3, one extra
reactor hour or one extra tonne of warehouse space?

**Exercise 4 (intermediate).** The plant can buy the Binder from a toll manufacturer at 78 USD/t, up to 40 t
per period, which uses no reactor time. Add a purchase variable `q_t >= 0` to the balance for the Binder and
re-solve. Report how much is bought, in which periods, and the change in total cost. At what toll price does
purchasing stop entirely? Find that price by a bisection search over at most 20 solves and verify it against
the reactor dual.

**Exercise 5 (intermediate).** Add a changeover restriction: the reactor cannot make both products in the
same period. Introduce a binary `y_{p,t}` with `x_{p,t} <= C_p y_{p,t}` and `sum_p y_{p,t} <= 1`, and solve
the resulting MILP with the same solver. Report the new cost and the increase over the LP. Explain why the LP
value is a valid lower bound on the MILP value.

**Exercise 6 (advanced).** Demand is uncertain. Build a two-stage stochastic version with three equally
likely demand scenarios for the Additive, obtained by scaling the given demand vector by 0.85, 1.00 and 1.15.
Production in period 1 must be the same in every scenario (a non-anticipativity constraint), while later
periods may adapt. Report the expected cost, the first-period production, and compare with the deterministic
plan built on mean demand. Quantify the value of the stochastic solution.

**Exercise 7 (introductory, cross-tool).** Build the single-product block of
`excel/W02_multiperiod_planning_STUDENT.xlsx` with the layout given above: `Solution` rows
`B5:G7`, objective cell `C13`, the balance `VALUE` row `B18:G18` against its zero `RHS` row, and the
capacity entry `$B$5:$G$5 <= $B$16:$G$16`. Solve it with OpenSolver using CBC. Confirm that cell `C13`
equals the optimal total cost printed in Section 3 to the cent and that row `B5:G5` equals the
production column of the results table period by period. Then extend the sheet to twelve periods by
repeating the demand, capacity and cost data, count every cell you had to create or copy, and state in
one sentence what the equivalent change is in the Pyomo model.


## 8. Takeaways

- The five stages of the Week 1 workflow are the format for every model in the modeling block. Stage 2, the structured specification with units on every symbol, is what makes stage 3 mechanical, and recording an assumption such as "everything produced is sold" as a numbered restriction is what makes it possible to challenge it later.
- Linear models are assembled from four patterns, not derived from scratch: an index set for each dimension, a balance equation for each conserved quantity, a capacity constraint for each scarce resource, and a linking constraint wherever two otherwise independent blocks meet. Recognizing the pattern is most of stage 3.
- A multiperiod model is defined by its linking constraint. The inventory balance `I_{t-1} - B_{t-1} + x_t - d_t = I_t - B_t` is the only place where consecutive periods meet, and everything the model can do about seasonality passes through it.
- Splitting the net stock position into non-negative inventory and backlog variables keeps the model linear and prices the two directions differently. Complementarity is enforced by the cost structure, not by binary variables.
- A horizon-end condition such as `B_{|T|} = 0` is mandatory. Without it the model exports its difficulties past the end of the horizon and reports a cost that cannot be achieved.
- Multi-product models separate into independent single-product problems unless a shared resource couples them. The coupling constraint is where the modeling value lies, and the difference between the coupled and uncoupled optima measures the cost of sharing.
- LP duals on capacity and resource constraints give the marginal value of debottlenecking, period by period. They are valid only locally: the value curve is piecewise linear and eventually flat, so duals must be re-computed after any large change.